In [ ]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "gemma2:2b",
        "prompt": "Hello",
        "stream": False 
    }
)

print(response.json())

{'model': 'gemma2:2b', 'created_at': '2026-05-07T04:08:48.5151865Z', 'response': 'Hello! 👋  \n\nHow can I help you today? 😄 \n', 'done': True, 'done_reason': 'stop', 'context': [106, 1645, 108, 4521, 107, 108, 106, 2516, 108, 4521, 235341, 169692, 139, 109, 2299, 798, 590, 1707, 692, 3646, 235336, 123781, 235248, 108], 'total_duration': 4098894200, 'load_duration': 329256900, 'prompt_eval_count': 10, 'prompt_eval_duration': 657276200, 'eval_count': 16, 'eval_duration': 2925682100}


In [11]:
import requests

url = "http://127.0.0.1:11434/api/generate"

payload = {
    "model": "gemma2:2b",
    "prompt": "có những loại giày nào?",
    "stream": False
}

response = requests.post(url, json=payload)

data = response.json()

print("AI:", data["response"])

AI: Tôi không thể cho bạn biết **phương pháp phân loại size giày** vì mỗi hãng giày sẽ sử dụng một quy định riêng. 

Tuy nhiên, để tìm hiểu thông tin về size giày, bạn có thể:

* **Kiểm tra trang web của các hãng giày yêu thích:**  Mọi hãng giày đều hiển thị size giày trên website của họ.
* **Tham khảo bảng size giày tại các cửa hàng:** Bạn có thể đến cửa hàng bán giày và xem bảng size để tìm hiểu cách đo size. 
* **Sử dụng công cụ tính kích thước size:** Một số trang web hoặc ứng dụng hỗ trợ tính toán size giày.

**Lưu ý**: Size giày là một khía cạnh quan trọng, nên bạn nên tham khảo nhiều nguồn thông tin để có được những thông tin chính xác nhất! 


Bạn muốn biết thêm về **size giày nào cụ thể?**  



In [13]:
from pymilvus import MilvusClient

client = MilvusClient(
    uri="http://localhost:19530"
)

print(client.get_server_version())

v2.4.7


In [3]:
import requests
import json
import sys

# =========================
# CẤU HÌNH HỆ THỐNG
# =========================
OLLAMA_URL = "http://127.0.0.1:11434/api/chat"
MODEL_NAME = "gemma2:2b"

# Dữ liệu tri thức (Sau này sẽ thay bằng bước truy vấn từ Milvus)
KNOWLEDGE_BASE = """
DANH SÁCH SẢN PHẨM:
1. Sofa Nordic màu kem: 8 triệu, hợp phòng nhỏ.
2. Sofa chữ L màu xám: 15 triệu, hiện đại.
3. Bàn làm việc gỗ MDF: 2 triệu, chống ẩm.
4. Ghế công thái học: 3 triệu, hỗ trợ cột sống.
5. Tủ quần áo gỗ sồi: 12 triệu.
6. Giường Scandinavian trắng: 10 triệu.
7. Kệ trang trí gỗ: 1.5 triệu.
8. Bàn ăn 4 ghế luxury: 18 triệu.
"""

SYSTEM_PROMPT = f"""
Bạn là nhân viên tư vấn bán hàng nội thất chuyên nghiệp.
Nhiệm vụ: Tư vấn nhiệt tình, lịch sự bằng tiếng Việt. 
Dựa vào dữ liệu sau để trả lời: {KNOWLEDGE_BASE}
Nếu khách hỏi sản phẩm không có trong danh sách, hãy khéo léo gợi ý sản phẩm tương tự.
"""

def chat_with_bot():
    # Khởi tạo lịch sử cuộc trò chuyện với System Prompt
    history = [
        {"role": "system", "content": SYSTEM_PROMPT}
    ]

    print("--- CHÀO MỪNG BẠN ĐẾN VỚI CỬA HÀNG NỘI THẤT AI ---")
    print("(Nhấn 'q' để thoát, 'c' để xóa lịch sử chat)")

    while True:
        print("\n" + "="*30)
        user_input = input("Khách hàng: ").strip()

        if user_input.lower() == 'q':
            print("AI: Cảm ơn quý khách. Hẹn gặp lại!")
            break
        
        if user_input.lower() == 'c':
            history = [{"role": "system", "content": SYSTEM_PROMPT}]
            print("--- Đã xóa lịch sử cuộc trò chuyện ---")
            continue

        if not user_input:
            continue

        # Thêm câu hỏi của khách vào lịch sử
        history.append({"role": "user", "content": user_input})

        payload = {
            "model": MODEL_NAME,
            "messages": history,
            "stream": True  # Bật streaming để chữ chạy dần dần
        }

        print("\nAI tư vấn: ", end="", flush=True)
        
        full_response = ""
        try:
            response = requests.post(OLLAMA_URL, json=payload, stream=True)
            response.raise_for_status()

            for line in response.iter_lines():
                if line:
                    chunk = json.loads(line.decode('utf-8'))
                    if 'message' in chunk:
                        content = chunk['message']['content']
                        print(content, end="", flush=True)
                        full_response += content
                    
                    if chunk.get('done'):
                        # Thêm câu trả lời của AI vào lịch sử để nhớ ngữ cảnh
                        history.append({"role": "assistant", "content": full_response})
                        print("\n")

        except requests.exceptions.ConnectionError:
            print("\nLỖI: Không thể kết nối với Ollama. Hãy chắc chắn bạn đã chạy 'ollama serve'.")
            break
        except Exception as e:
            print(f"\nLỖI: {e}")
            break

if __name__ == "__main__":
    chat_with_bot()

--- CHÀO MỪNG BẠN ĐẾN VỚI CỬA HÀNG NỘI THẤT AI ---
(Nhấn 'q' để thoát, 'c' để xóa lịch sử chat)


AI tư vấn: Chào bạn! 😊  Để tìm sản phẩm phù hợp, bạn muốn thử  một vài món đồ nội thất khác nhé?

Bạn có thể tham khảo:

* **Bàn làm việc gỗ MDF:** Với giá chỉ 2 triệu, bạn có thể dễ dàng trang bị cho phòng làm việc của mình một chiếc bàn chất lượng.
* **Ghế công thái học:** Ghế công thái học hỗ trợ cột sống với giá chỉ 3 triệu sẽ giúp bạn thoải mái làm việc và nghỉ ngơi.

Chúc bạn tìm thấy sản phẩm ưng ý! 😊  




AI tư vấn: Ah, mình thấy bạn đang xem xét những dòng sofa sang trọng. 👍 Chào bạn, mình có thể tư vấn một số sản phẩm phù hợp với mức ngân sách của bạn:

* **Sofa chữ L màu xám:**  Với mức giá 15 triệu, chiếc sofa chữ L này mang đến vẻ hiện đại cho không gian của bạn.
* **Bàn ăn 4 ghế luxury:** Với mức giá 18 triệu, chiếc bàn ăn này sẽ mang đến nét sang trọng và tiện nghi cho bữa ăn gia đình.

Mình rất mong bạn có thể trải nghiệm những sản phẩm tuyệt vời! 💫  





AI: Cảm ơn quý k